In [2]:
import pandas as pd
from tqdm.notebook import tqdm_notebook as tqdm
import os
import itertools
from rsa import compute_rsa
import gc

In [3]:
names = [name.split('.')[0] for name in os.listdir('../../data/rsms') if name.endswith('.pkl')]
name_combs = list(itertools.combinations(names, 2))
name_combs = [tuple(sorted(pair)) for pair in name_combs]
len(name_combs)

435

In [4]:
def get_rsa_workload(results_path, all_combinations):
    """
    Checks for an RSA results file. If found, it determines which
    combinations still need to be computed. If not found, it creates an
    empty file and recursively calls itself.
    """

    # Base Case: The file exists.
    if os.path.exists(results_path):
        print(f"✔️ Results file found at '{results_path}'")
        existing_results_df = pd.read_csv(results_path)

        if existing_results_df.empty:
            already_computed = set()
        else:
            # Create a set of tuples for efficient lookup
            already_computed = existing_results_df[['name_i', 'name_j']].apply(tuple, axis=1).tolist()
            already_computed = set(tuple(sorted(pair)) for pair in already_computed)

        # Use set difference to find what's left to compute
        to_compute = list(set(all_combinations) - already_computed)
        return to_compute, existing_results_df

    # Recursive Step: The file does not exist.
    else:
        print(f"⚠️ Results file not found. Creating empty file...")
        # Ensure the parent directory exists
        os.makedirs(os.path.dirname(results_path), exist_ok=True)

        # Create and save an empty DataFrame with the correct headers
        columns = ['name_i', 'name_j', 'spearman', 'n_words']
        pd.DataFrame(columns=columns).to_csv(results_path, index=False)

        # Now that the file exists, call the function again
        return get_rsa_workload(results_path, all_combinations)


results_dir = '../../data/results'
results_file = 'rsa.csv'
results_file_full_path = os.path.join(results_dir, results_file)

to_compute, old_results = get_rsa_workload(results_file_full_path, name_combs)
to_compute

✔️ Results file found at '../../data/results/rsa.csv'


[('Llama_3_8B', 'microarray'),
 ('BERT', 'spherical_text_Wikipedia'),
 ('BERT', 'microarray'),
 ('Llama_3_8B', 'morphoNLM'),
 ('BERT', 'PPMI_SVD_SouthFlorida'),
 ('BERT_large_lenci', 'microarray'),
 ('BERT_large_lenci', 'CBOW_GoogleNews'),
 ('EEG_text', 'Llama_3_8B'),
 ('Llama_3_8B', 'SGSoftMaxOutput_SWOW'),
 ('BERT_large_lenci', 'EEG_text'),
 ('BERT_large_lenci', 'morphoNLM'),
 ('BERT_large_lenci', 'SGSoftMaxOutput_SWOW'),
 ('BERT_large_lenci', 'GloVe_Wikipedia'),
 ('GloVe_Twitter', 'Llama_3_8B'),
 ('Llama_3_8B', 'fMRI_text_cognival'),
 ('BERT', 'fMRI_text_cognival'),
 ('BERT_large_lenci', 'PPMI_SVD_SWOW'),
 ('BERT_large_lenci', 'EEG_speech'),
 ('GloVe_CommonCrawl', 'Llama_3_8B'),
 ('Llama_3_8B', 'SGSoftMaxInput_SWOW'),
 ('BERT_large_lenci', 'feature_overlap'),
 ('BERT', 'CBOW_GoogleNews'),
 ('BERT', 'SGSoftMaxInput_SWOW'),
 ('BERT', 'EEG_text'),
 ('BERT', 'morphoNLM'),
 ('BERT_large_lenci', 'SGSoftMaxInput_SWOW'),
 ('BERT', 'SGSoftMaxOutput_SWOW'),
 ('BERT', 'GloVe_Wikipedia'),
 ('Ll

In [5]:
# Function to process each file pair
def process_file_pair(f_name_i, f_name_j, dir_path):
    
    # Load the RSMs
    rsm_i = pd.read_pickle(dir_path + f_name_i)
    rsm_j = pd.read_pickle(dir_path + f_name_j)

    # Compute RSA. If len(intersection) > 10000, randomly sample 10000 words
    corr, n_words = compute_rsa(rsm_i, rsm_j, max_n=10000)
    print(f"spearman_r={corr}, n_words={n_words}")
    print('-------------------------------------------------')
    
    # Free memory by deleting the RSMs
    del rsm_i, rsm_j
    gc.collect()

    return corr, n_words


# Compute RSA in parallel
rsm_dir_path = f'../../data/rsms/'
new_results = []
for name_i, name_j in tqdm(to_compute):
    print(f"{name_i, name_j}")
    spearman, n = process_file_pair(f'{name_i}.pkl', f'{name_j}.pkl', rsm_dir_path)
    new_results.append([name_i, name_j, spearman, n])
    
# Save the results
new_results = pd.DataFrame(new_results, columns=['name_i', 'name_j', 'spearman', 'n_words'])
results = pd.concat([old_results, new_results], ignore_index=True)
results

  0%|          | 0/84 [00:00<?, ?it/s]

('Llama_3_8B', 'microarray')
spearman_r=-0.03069566579395403, n_words=403
-------------------------------------------------
('BERT', 'spherical_text_Wikipedia')
spearman_r=0.3164651575514117, n_words=9866
-------------------------------------------------
('BERT', 'microarray')
spearman_r=0.11090514330726452, n_words=403
-------------------------------------------------
('Llama_3_8B', 'morphoNLM')
spearman_r=0.30007685835595865, n_words=9737
-------------------------------------------------
('BERT', 'PPMI_SVD_SouthFlorida')
spearman_r=0.11610239358787959, n_words=4120
-------------------------------------------------
('BERT_large_lenci', 'microarray')
spearman_r=0.11731785348622221, n_words=254
-------------------------------------------------
('BERT_large_lenci', 'CBOW_GoogleNews')
spearman_r=0.1808117505182261, n_words=14336
-------------------------------------------------
('EEG_text', 'Llama_3_8B')
spearman_r=-0.036367691041888764, n_words=1680
--------------------------------------

KeyboardInterrupt: 

In [ ]:
results.to_csv(results_file_full_path, index=False)